# LangSmith trace explorer

Download **runs**, **trace trees**, and **conversation threads** from LangSmith and save them as JSON under `langsmith/exports/`.

**Setup:** needs `LANGSMITH_API_KEY` (and `LANGSMITH_PROJECT`) in your project `.env`. The `langsmith` SDK ships with the project deps.

Run **Setup** first, use **Browse** to find IDs, then run whichever export cell you need (paste the id into the UPPER_CASE variable at the top of the cell).

## Setup

In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv
from langsmith import Client

load_dotenv()  # LANGSMITH_API_KEY / LANGSMITH_PROJECT from .env

client = Client()
PROJECT = 'bank-python'           # your LANGSMITH_PROJECT
EXPORT_DIR = Path('exports')
EXPORT_DIR.mkdir(exist_ok=True)

def to_dict(run):
    # Best-effort convert a LangSmith Run object to a plain dict.
    for attr in ('model_dump', 'dict'):
        if hasattr(run, attr):
            try:
                return getattr(run, attr)()
            except Exception:
                pass
    return json.loads(run.json())

def save_json(data, name):
    path = EXPORT_DIR / (name + '.json')
    path.write_text(json.dumps(data, indent=2, default=str))
    print('saved', path, '(' + str(path.stat().st_size) + ' bytes)')
    return path

## Browse recent runs (find run_id / trace_id)

In [2]:
recent = list(client.list_runs(project_name=PROJECT, limit=20))
for r in recent:
    print(r.start_time, '|', getattr(r, 'run_type', '?'), '|', r.name,
          '| run_id:', r.id, '| trace_id:', r.trace_id)

2026-06-25 11:41:46.638584+00:00 | parser | PydanticToolsParser | run_id: 019efe96-0b0e-7b62-a66f-29e9f7d88e42 | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:46.637422+00:00 | chain | _raise_if_no_tool_calls | run_id: 019efe96-0b0d-7122-b666-1ed05a3748e1 | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:27.322418+00:00 | llm | ChatAnthropic | run_id: 019efe95-bf9a-7821-92bd-c1b63cba2fa1 | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:27.321186+00:00 | chain | Judge | run_id: 019efe95-bf99-7c63-971e-6479f18df5ea | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:26.596728+00:00 | tool | query_transactions | run_id: 019efe95-bcc4-7d53-99b0-0c87f283519d | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:26.223908+00:00 | tool | query_balance | run_id: 019efe95-bb4f-71c1-bcb2-c4dd43d19d68 | trace_id: 019efe95-90b8-7c11-8dc7-b0d759304b50
2026-06-25 11:41:26.222580+00:00 | parser | PydanticToolsParser | run_id

## 1. A single run by run_id

In [ ]:
RUN_ID = 'paste-a-run-id-here'

run = client.read_run(RUN_ID)
save_json(to_dict(run), 'run-' + RUN_ID)

## 2. A full trace tree (root + all child runs) by trace_id

In [ ]:
TRACE_ID = 'paste-a-trace-id-here'

runs = list(client.list_runs(project_name=PROJECT, trace_id=TRACE_ID))
# Fallback if your SDK lacks the trace_id kwarg:
# runs = list(client.list_runs(project_name=PROJECT, filter='eq(trace_id, "' + TRACE_ID + '")'))
print(len(runs), 'runs in trace')
save_json([to_dict(r) for r in runs], 'trace-' + TRACE_ID)

## 3. A conversation thread (all runs sharing `session_id`)

`ask()` / the probe tag each turn with `metadata.session_id = thread_id`, so a thread is just the runs whose `session_id` matches.

In [4]:
THREAD_ID = 'bc4ab30f-fc3d-40e0-9602-3a30980f2ba8'

flt = 'and(eq(metadata_key, "session_id"), eq(metadata_value, "' + THREAD_ID + '"))'
runs = list(client.list_runs(project_name=PROJECT, filter=flt))
print(len(runs), 'runs in thread')
save_json([to_dict(r) for r in runs], 'thread-' + THREAD_ID)

15 runs in thread
saved exports/thread-bc4ab30f-fc3d-40e0-9602-3a30980f2ba8.json (106825 bytes)


PosixPath('exports/thread-bc4ab30f-fc3d-40e0-9602-3a30980f2ba8.json')